* **구현 목적**: 
  - AI-Hub 데이터셋에는 프로젝트 범위(Vision)를 벗어나는 LiDAR 센서 데이터가 혼재되어 있음.
  - 학습 전, **LiDAR 디렉토리를 탐색에서 제외(Exclude)**하고 순수 이미지(JPG)와 라벨(JSON)의 1:1 매칭 여부를 검증함.
* **주요 로직**:
  - `os.walk`를 이용해 디렉토리를 순회하되 `dirs.remove('Lidar')`를 통해 불필요한 데이터를 원천 차단.
  - Set(집합) 자료구조를 활용하여 파일명 매칭 연산을 수행, 누락된 데이터(Missing Data)를 사전에 필터링.

In [ ]:
import os

# =========================================================
# [설정] 검증할 두 개의 최상위 폴더 경로
# =========================================================
JSON_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\라벨링데이터"
IMAGE_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\원천데이터"

def get_filename_set(root_dir, target_ext):
    """
    디렉토리를 순회하되, 'Lidar' 폴더는 건너뜁니다.
    순수 파일명만 수집하여 반환합니다.
    """
    file_names = set()
    file_paths = {} 
    
    print(f"📂 스캔 중: {root_dir} (확장자: {target_ext})...")
    
    skipped_lidar_count = 0
    count = 0
    
    for root, dirs, files in os.walk(root_dir):
        # -----------------------------------------------------------
        # [핵심 수정] 'Lidar' 디렉토리가 있으면 탐색 목록에서 제외
        # -----------------------------------------------------------
        # 대소문자 구분 없이 'lidar'가 포함되면 제외하려면 .lower() 사용
        # 여기서는 정확히 'Lidar'라는 폴더를 제외합니다.
        if 'Lidar' in dirs:
            dirs.remove('Lidar')
            skipped_lidar_count += 1
            # print(f"   ㄴ [제외됨] Lidar 폴더 건너뜀")
            
        for filename in files:
            if filename.lower().endswith(target_ext):
                # 1. 확장자 제거 (순수 파일명 추출)
                name_only = os.path.splitext(filename)[0]
                
                # 2. 파일명 등록
                file_names.add(name_only)
                file_paths[name_only] = os.path.join(root, filename)
                count += 1
                
    print(f"   ㄴ 발견된 파일 수: {count}개 (Lidar 폴더 {skipped_lidar_count}개 제외됨)")
    return file_names, file_paths

def verify_dataset_ignore_lidar(json_dir, img_dir):
    # 1. 파일명 수집 (Lidar 제외)
    json_names, json_paths = get_filename_set(json_dir, '.json')
    img_names, img_paths = get_filename_set(img_dir, '.jpg')
    
    # 2. 비교 연산
    common = json_names & img_names         # 교집합 (매칭 성공)
    only_json = json_names - img_names      # JSON만 있음
    only_img = img_names - json_names       # 이미지만 있음
    
    print("\n" + "="*50)
    print("📊 [검증 결과 리포트 (Lidar 제외)]")
    print("="*50)
    
    print(f"✅ 1:1 매칭 성공: {len(common)}쌍")
    
    # 3. 불일치 내용 상세 출력
    if len(only_json) > 0:
        print(f"\n❌ [경고] 짝꿍 이미지(.jpg)가 없는 JSON 파일: {len(only_json)}개")
        for i, name in enumerate(list(only_json)[:5]):
            print(f"   - {name}.json (위치: {json_paths[name]})")
        if len(only_json) > 5: print("   ... (생략)")
            
    if len(only_img) > 0:
        print(f"\n❌ [경고] 짝꿍 JSON(.json)이 없는 이미지 파일: {len(only_img)}개")
        for i, name in enumerate(list(only_img)[:5]):
            print(f"   - {name}.jpg (위치: {img_paths[name]})")
        if len(only_img) > 5: print("   ... (생략)")

    # 4. 최종 결론
    print("\n" + "="*50)
    if len(only_json) == 0 and len(only_img) == 0:
        print("🎉 축하합니다! (Lidar를 제외한) 모든 데이터가 완벽하게 1:1 매칭됩니다.")
    else:
        print("⚠️ 불일치가 발견되었습니다. 위 리스트를 확인하세요.")

# 실행
if os.path.exists(JSON_ROOT_DIR) and os.path.exists(IMAGE_ROOT_DIR):
    verify_dataset_ignore_lidar(JSON_ROOT_DIR, IMAGE_ROOT_DIR)
else:
    print("경로가 잘못되었습니다. 폴더 주소를 확인해주세요.")

본 프로젝트는 **단안 카메라(Vision)** 기반 시스템이므로, 용량을 많이 차지하는 LiDAR(Point Cloud) 데이터는 불필요함.
  - 학습 데이터셋 구축 전, 'Lidar' 폴더를 일괄 삭제하여 저장 공간을 확보하고 데이터 로딩 효율을 높임.

In [ ]:
import os
import shutil  # 폴더를 통째로 지울 때 사용하는 라이브러리

# =========================================================
# [설정] Lidar 폴더를 삭제할 두 개의 최상위 폴더 경로
# =========================================================
JSON_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\라벨링데이터"
IMAGE_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\원천데이터"

# True로 바꾸면 실제로 삭제됩니다. (처음엔 False로 테스트 권장)
DELETE_MODE = True

def remove_lidar_directories(root_dir):
    print(f"🚀 스캔 시작: {root_dir}")
    lidar_paths = []

    # 1. 삭제할 Lidar 폴더 위치 모두 찾기
    for root, dirs, files in os.walk(root_dir):
        for dirname in dirs:
            # 대소문자 구분 없이 'lidar' 이름이 정확히 일치하면 타겟
            if dirname.lower() == 'lidar':
                full_path = os.path.join(root, dirname)
                lidar_paths.append(full_path)

    # 2. 찾은 폴더 삭제하기
    if not lidar_paths:
        print("   ㄴ 'Lidar' 폴더를 하나도 발견하지 못했습니다.")
    else:
        print(f"   ㄴ 총 {len(lidar_paths)}개의 Lidar 폴더를 발견했습니다.")
        
        for path in lidar_paths:
            if DELETE_MODE:
                try:
                    shutil.rmtree(path) # 폴더와 그 안의 내용물까지 강제 삭제
                    print(f"      [삭제 완료] {path}")
                except Exception as e:
                    print(f"      [삭제 실패] {path} - 이유: {e}")
            else:
                print(f"      [삭제 예정] {path}")

    print("-" * 50)

# 실행
print("=" * 50)
print("🛑 Lidar 폴더 삭제 작업 (Lidar Removal)")
print("=" * 50)

if os.path.exists(JSON_ROOT_DIR) and os.path.exists(IMAGE_ROOT_DIR):
    # JSON 폴더 청소
    remove_lidar_directories(JSON_ROOT_DIR)
    
    # 이미지 폴더 청소
    remove_lidar_directories(IMAGE_ROOT_DIR)
    
    if not DELETE_MODE:
        print("\n※ 현재 '안전 모드'입니다. 로그를 확인하고 이상 없으면")
        print("   코드 상단의 DELETE_MODE = True 로 변경해서 진짜로 지우세요!")
    else:
        print("\n✨ 모든 Lidar 폴더 삭제가 완료되었습니다. 데이터셋이 아주 깔끔해졌습니다!")
else:
    print("경로가 잘못되었습니다. 폴더 주소를 확인해주세요.")

Class 오버레이해서 어떤 파일인지 확인 

In [ ]:
import os
import json
import shutil
from PIL import Image, ImageDraw, ImageFont

# =========================================================
# [설정] 경로
# =========================================================
JSON_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\라벨링데이터"
IMAGE_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\원천데이터"
SAVE_ROOT_DIR = r"E:\파이썬 공학경진대회\Extract_Test_Images"

# =========================================================
# [설정] 사용자 요청 13개 타겟 클래스 (확정)
# =========================================================
TARGET_CLASSES = [
    "bollard",
    "truck",
    "motorcycle",
    "baricade",        # 오타 유지 (baricade)
    "kickboard",
    "Warning Block",   # 대문자
    "car",
    "drainage",
    "manhole",
    "standing banner",
    "Leading Block",   # 대문자
    "labacon",
    "bicycle"
]

# 각 클래스별 고유 색상 지정 (RGB)
CLASS_COLORS = {
    "bollard": (255, 0, 0),          # 빨강
    "truck": (75, 0, 130),           # 남색 (Indigo)
    "motorcycle": (0, 0, 128),       # 네이비
    "baricade": (255, 165, 0),       # 주황
    "kickboard": (255, 20, 147),     # 딥핑크
    "Warning Block": (255, 215, 0),  # 골드 (노란색 계열)
    "car": (0, 128, 128),            # 틸 (Teal)
    "drainage": (0, 0, 255),         # 파랑
    "manhole": (138, 43, 226),       # 보라
    "standing banner": (0, 255, 255),# 하늘색
    "Leading Block": (128, 0, 128),  # 자주
    "labacon": (255, 255, 0),        # 노랑
    "bicycle": (0, 255, 0)           # 라임
}

# 각 클래스별 5장씩만 추출
MAX_COUNT_PER_CLASS = 5

# ---------------------------------------------------------
# [함수] 이미지 위에 라벨 오버레이 그리기
# ---------------------------------------------------------
def draw_overlay(image_path, json_data, target_labels):
    try:
        original_image = Image.open(image_path).convert("RGBA")
    except Exception as e:
        print(f"⚠️ 이미지 열기 실패: {image_path} - {e}")
        return None

    overlay_layer = Image.new("RGBA", original_image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay_layer)

    try:
        font = ImageFont.truetype("arial.ttf", 24) # 폰트 크기 약간 키움
    except IOError:
        font = ImageFont.load_default()

    shapes = json_data.get('shapes', [])
    for shape in shapes:
        label = shape.get('label')
        if label not in target_labels:
            continue
        
        points = shape.get('points')
        if not points:
            continue
            
        polygon_points = [tuple(point) for point in points]
        
        # 색상 설정 (매핑 안 된 라벨은 회색 처리)
        color_rgb = CLASS_COLORS.get(label, (128, 128, 128))
        
        fill_color = color_rgb + (80,)   # 투명도 80 (약하게 채우기)
        outline_color = color_rgb + (255,) # 불투명 외곽선
        
        draw.polygon(polygon_points, fill=fill_color, outline=outline_color, width=4)
        
        # 텍스트 그리기
        text_pos = polygon_points[0]
        try:
            text_bbox = draw.textbbox(text_pos, label, font=font)
            draw.rectangle(text_bbox, fill=(0, 0, 0, 180))
        except AttributeError:
             draw.rectangle((text_pos[0], text_pos[1], text_pos[0]+100, text_pos[1]+25), fill=(0,0,0,180))

        draw.text(text_pos, label, font=font, fill=(255, 255, 255, 255))

    combined_image = Image.alpha_composite(original_image, overlay_layer)
    return combined_image.convert("RGB")

# ---------------------------------------------------------
# [메인 함수]
# ---------------------------------------------------------
def extract_and_overlay_images():
    # 1. 이미지 위치 인덱싱
    print("🔍 1단계: 원천 데이터 폴더 스캔 중... (이미지 위치 파악)")
    image_map = {}
    image_count = 0
    for root, dirs, files in os.walk(IMAGE_ROOT_DIR):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                name_no_ext = os.path.splitext(file)[0]
                image_map[name_no_ext] = os.path.join(root, file)
                image_count += 1
    
    print(f"✅ 이미지 인덱싱 완료: 총 {image_count}개의 이미지를 찾았습니다.")
    if image_count == 0:
        print("❌ 오류: 원천 데이터에 이미지가 없습니다.")
        return

    # 2. 작업 시작
    collected_counts = {cls: 0 for cls in TARGET_CLASSES}
    if not os.path.exists(SAVE_ROOT_DIR):
        os.makedirs(SAVE_ROOT_DIR)

    print(f"\n🚀 2단계: 13개 타겟 라벨 오버레이 시작...")

    for root, dirs, files in os.walk(JSON_ROOT_DIR):
        if all(value >= MAX_COUNT_PER_CLASS for value in collected_counts.values()):
            print("\n✨ 모든 클래스 수집 완료! 작업을 종료합니다.")
            break

        for file in files:
            if not file.lower().endswith('.json'):
                continue
            
            json_name_no_ext = os.path.splitext(file)[0]
            if json_name_no_ext not in image_map:
                continue

            try:
                json_path = os.path.join(root, file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                labels_to_save = set()
                shapes = data.get('shapes', [])
                for shape in shapes:
                    label = shape.get('label')
                    # 정확히 사용자가 요청한 리스트에 포함되는지 확인
                    if label in TARGET_CLASSES and collected_counts[label] < MAX_COUNT_PER_CLASS:
                        labels_to_save.add(label)
                
                if labels_to_save:
                    real_image_path = image_map[json_name_no_ext]
                    image_filename = os.path.basename(real_image_path)
                    
                    # 오버레이 생성
                    overlayed_image = draw_overlay(real_image_path, data, TARGET_CLASSES)
                    
                    if overlayed_image:
                        saved_labels_in_loop = set()
                        for label in labels_to_save:
                            if label in saved_labels_in_loop:
                                continue

                            save_dir = os.path.join(SAVE_ROOT_DIR, label)
                            if not os.path.exists(save_dir):
                                os.makedirs(save_dir)
                                
                            save_path = os.path.join(save_dir, image_filename)
                            overlayed_image.save(save_path, "JPEG", quality=95)
                            
                            collected_counts[label] += 1
                            print(f"[{label}] ({collected_counts[label]}/{MAX_COUNT_PER_CLASS}) 저장: {image_filename}")
                            saved_labels_in_loop.add(label)

            except Exception as e:
                print(f"⚠️ 에러 ({file}): {e}")

    # 리포트
    print("\n" + "="*50)
    print("📊 최종 결과 (각 5장 목표)")
    print("="*50)
    for cls, count in collected_counts.items():
        status = "완료" if count >= MAX_COUNT_PER_CLASS else "부족"
        print(f" - {cls:<15}: {count}장 ({status})")
    
    print("-" * 50)
    print(f"📂 저장 경로: {SAVE_ROOT_DIR}")

if __name__ == "__main__":
    extract_and_overlay_images()

* **구현 목적**:
  - 13개 타겟 클래스(Target Classes)에 대한 데이터 개수를 정량적으로 파악.
  - 'Bollard'와 같이 과도하게 많은 클래스와 'Truck'처럼 부족한 클래스 간의 **불균형(Imbalance)을 시각화**하여, 추후 오버샘플링(Oversampling) 전략 수립의 근거로 활용

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from collections import Counter

# =========================================================
# [설정] 경로
# =========================================================
JSON_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\라벨링데이터"
IMAGE_ROOT_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\원천데이터"

# =========================================================
# [설정] 타겟 클래스 (13개)
# =========================================================
TARGET_CLASSES = [
    "bollard", "truck", "motorcycle", "baricade", "kickboard",
    "Warning Block", "car", "drainage", "manhole", "standing banner",
    "Leading Block", "labacon", "bicycle"
]

# 그래프용 색상 (이전 코드와 동일하게 매칭)
CLASS_COLORS = {
    "bollard": "#FF0000",          # 빨강
    "truck": "#4B0082",            # 남색
    "motorcycle": "#000080",       # 네이비
    "baricade": "#FFA500",         # 주황
    "kickboard": "#FF1493",        # 딥핑크
    "Warning Block": "#FFD700",    # 골드
    "car": "#008080",              # 틸
    "drainage": "#0000FF",         # 파랑
    "manhole": "#8A2BE2",          # 보라
    "standing banner": "#00FFFF",  # 하늘색
    "Leading Block": "#800080",    # 자주
    "labacon": "#FFFF00",          # 노랑
    "bicycle": "#00FF00"           # 라임
}

def analyze_dataset():
    print("🔍 1단계: 원천 데이터(이미지) 인덱싱 중...")
    image_map = set() # 존재 여부만 확인하면 되므로 set 사용
    for root, dirs, files in os.walk(IMAGE_ROOT_DIR):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                name_no_ext = os.path.splitext(file)[0]
                image_map.add(name_no_ext)
    
    print(f"✅ 이미지 총 {len(image_map)}개 인덱싱 완료.")

    print("\n🔍 2단계: 전체 JSON 스캔 및 라벨 카운팅 시작...")
    
    # 1. 객체 수 (Instance Count): 라벨이 등장한 총 횟수
    instance_counts = Counter()
    # 2. 이미지 수 (Image Count): 해당 라벨이 포함된 파일 개수
    image_counts = Counter()
    
    processed_files = 0

    for root, dirs, files in os.walk(JSON_ROOT_DIR):
        for file in files:
            if not file.lower().endswith('.json'):
                continue
            
            # 이미지가 존재하는 JSON만 분석 (유효 데이터)
            name_no_ext = os.path.splitext(file)[0]
            if name_no_ext not in image_map:
                continue

            processed_files += 1
            if processed_files % 5000 == 0:
                print(f"   ... {processed_files}개 파일 분석 중")

            try:
                path = os.path.join(root, file)
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                shapes = data.get('shapes', [])
                
                # 현재 파일에 등장한 라벨 집합 (이미지 수 카운트용)
                labels_in_this_file = set()

                for shape in shapes:
                    label = shape.get('label')
                    
                    if label in TARGET_CLASSES:
                        instance_counts[label] += 1
                        labels_in_this_file.add(label)
                
                # 이미지 수 업데이트
                for label in labels_in_this_file:
                    image_counts[label] += 1

            except Exception as e:
                print(f"⚠️ 에러 ({file}): {e}")

    print("-" * 60)
    print(f"✅ 분석 완료! (총 {processed_files}개의 유효한 데이터 쌍 분석)")
    print("-" * 60)
    
    # 텍스트 결과 출력
    print(f"{'Label Name':<20} | {'Instance Count (객체 수)':<25} | {'Image Count (이미지 수)':<20}")
    print("-" * 70)
    # 많이 등장한 순서대로 정렬
    sorted_labels = sorted(TARGET_CLASSES, key=lambda x: instance_counts[x], reverse=True)
    
    for label in sorted_labels:
        print(f"{label:<20} | {instance_counts[label]:<25,} | {image_counts[label]:<20,}")
    print("-" * 70)

    # =========================================================
    # 📊 히스토그램 시각화 (객체 수 기준)
    # =========================================================
    plt.figure(figsize=(14, 7))
    
    # X축: 라벨 이름, Y축: 개수
    x = list(range(len(sorted_labels)))
    y = [instance_counts[label] for label in sorted_labels]
    colors = [CLASS_COLORS.get(label, 'gray') for label in sorted_labels]
    
    bars = plt.bar(x, y, color=colors, edgecolor='black', alpha=0.8)
    
    # 그래프 꾸미기
    plt.title(f'Dataset Distribution (Total Objects: {sum(y):,})', fontsize=16, fontweight='bold')
    plt.xlabel('Class Names', fontsize=12)
    plt.ylabel('Number of Instances (Objects)', fontsize=12)
    plt.xticks(x, sorted_labels, rotation=45, ha='right', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    # 막대 위에 숫자 표시
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + (max(y)*0.01),
                 f'{int(height):,}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_dataset()

희귀 클래스(오토바이, 트럭, 킥보드 등)가 포함된 사진: 👉 100% 생존 (절대 지우지 않음)

흔한 클래스(바리케이드, 볼라드)만 있는 사진: 👉 10%만 랜덤 추출 (나머지 90% 폐기)

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

# =========================================================
# [설정] 경로
# =========================================================
SRC_DIR = r"E:\파이썬 공학경진대회\YOLO_Safety_Dataset"
DST_DIR = r"E:\파이썬 공학경진대회\YOLO_Safety_Dataset_Lite"

# [설정] 보존할 희귀 클래스 ID (이게 하나라도 있으면 무조건 저장)
# 1:Truck, 2:Motorcycle, 4:Kickboard, 12:Bicycle
RARE_CLASSES = [1, 2, 4, 12]

# [설정] 흔한 데이터 샘플링 비율 (10%만 남기기)
COMMON_KEEP_RATIO = 0.1 

def create_lite_dataset():
    print("✂️ 데이터셋 다이어트 시작...")
    
    # 1. Lite 폴더 구조 생성
    if os.path.exists(DST_DIR):
        try:
            shutil.rmtree(DST_DIR)
        except:
            pass
    
    for split in ['train', 'val']:
        os.makedirs(os.path.join(DST_DIR, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(DST_DIR, 'labels', split), exist_ok=True)

    # data.yaml 복사
    shutil.copy2(os.path.join(SRC_DIR, 'data.yaml'), os.path.join(DST_DIR, 'data.yaml'))

    # 2. Train 데이터 필터링 (Val은 건드리지 않고 다 가져감!)
    src_lbl_dir = os.path.join(SRC_DIR, 'labels', 'train')
    src_img_dir = os.path.join(SRC_DIR, 'images', 'train')
    
    dst_lbl_dir = os.path.join(DST_DIR, 'labels', 'train')
    dst_img_dir = os.path.join(DST_DIR, 'images', 'train')

    txt_files = [f for f in os.listdir(src_lbl_dir) if f.endswith(".txt")]
    
    kept_count = 0
    
    for txt_file in tqdm(txt_files, desc="Filtering Train Data"):
        txt_path = os.path.join(src_lbl_dir, txt_file)
        
        # 라벨 파일 분석
        is_rare = False
        with open(txt_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                cls_id = int(line.split()[0])
                if cls_id in RARE_CLASSES:
                    is_rare = True
                    break
        
        # [결정 로직]
        # 희귀 클래스가 있거나 OR 운 좋게 당첨된 경우만 저장
        should_keep = is_rare or (random.random() < COMMON_KEEP_RATIO)
        
        if should_keep:
            # 이미지 파일 찾기
            img_name = os.path.splitext(txt_file)[0]
            img_file = None
            for ext in ['.jpg', '.png', '.jpeg']:
                temp = os.path.join(src_img_dir, img_name + ext)
                if os.path.exists(temp):
                    img_file = img_name + ext
                    break
            
            if img_file:
                # 복사
                shutil.copy2(os.path.join(src_img_dir, img_file), os.path.join(dst_img_dir, img_file))
                shutil.copy2(txt_path, os.path.join(dst_lbl_dir, txt_file))
                kept_count += 1

    # 3. Validation 데이터는 100% 복사 (검증은 정확해야 하므로)
    print("📋 Validation 데이터 전체 복사 중...")
    shutil.copytree(os.path.join(SRC_DIR, 'images', 'val'), os.path.join(DST_DIR, 'images', 'val'), dirs_exist_ok=True)
    shutil.copytree(os.path.join(SRC_DIR, 'labels', 'val'), os.path.join(DST_DIR, 'labels', 'val'), dirs_exist_ok=True)

    print("\n" + "="*50)
    print(f"🎉 다이어트 완료!")
    print(f"   - 원본 학습 데이터: {len(txt_files)}장")
    print(f"   - Lite 학습 데이터: {kept_count}장")
    print(f"   - 감소율: 약 {100 - (kept_count/len(txt_files)*100):.1f}% 감소")
    print(f"📂 저장 위치: {DST_DIR}")
    print("="*50)

if __name__ == "__main__":
    create_lite_dataset()

희귀 데이터 물리적 증식 코드 

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

# =========================================================
# [설정] 분석할 데이터셋 경로 (Lite 폴더)
# =========================================================
DATASET_DIR = r"E:\파이썬 공학경진대회\YOLO_Safety_Dataset_Lite"
LABEL_DIR = os.path.join(DATASET_DIR, "labels", "train")
IMAGE_DIR = os.path.join(DATASET_DIR, "images", "train")

# [설정] 클래스 ID와 이름 매핑 (그래프에 이름으로 표시하기 위함)
# 사용자의 데이터셋 클래스 정의에 맞춰 수정 가능
CLASS_NAMES = {
    0: "Person",
    1: "Truck",       # 증식 대상
    2: "Motorcycle",  # 증식 대상 (핵심)
    3: "Car",
    4: "Kickboard",   # 증식 대상
    5: "Bus",
    6: "Bollard",
    7: "Curstone",
    8: "Crosswalk",
    9: "Pole",
    10: "Tree_trunk",
    11: "Dog",
    12: "Bicycle",    # 증식 대상
    # ... 필요한 만큼 추가
}

def analyze_dataset():
    # 1. 파일 존재 확인
    if not os.path.exists(LABEL_DIR):
        print(f"❌ 경로를 찾을 수 없습니다: {LABEL_DIR}")
        return

    txt_files = glob.glob(os.path.join(LABEL_DIR, "*.txt"))
    img_files = glob.glob(os.path.join(IMAGE_DIR, "*.*")) # jpg, png 등

    print(f"🧐 데이터셋 분석 중... (경로: {DATASET_DIR})")
    print(f"   - 총 이미지 파일 수: {len(img_files)}장")
    print(f"   - 총 라벨 파일 수: {len(txt_files)}개")
    
    # 2. 클래스별 객체 수 카운트
    class_counts = Counter()
    
    for txt_file in tqdm(txt_files, desc="Reading Labels"):
        with open(txt_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1

    # 3. 결과 출력 (터미널)
    print("\n" + "="*40)
    print("📊 클래스별 객체 수 (Instance Counts)")
    print("="*40)
    
    # 클래스 ID 순서대로 정렬하여 출력
    sorted_ids = sorted(class_counts.keys())
    
    plot_names = []
    plot_counts = []

    for cls_id in sorted_ids:
        count = class_counts[cls_id]
        name = CLASS_NAMES.get(cls_id, f"Class_{cls_id}")
        
        print(f"ID {cls_id:2d} [{name:^12}]: {count:6d} 개")
        
        plot_names.append(name)
        plot_counts.append(count)

    print("="*40)
    print(f"✅ 총 객체(Box) 수: {sum(class_counts.values())} 개")

    # 4. 시각화 (막대 그래프)
    plt.figure(figsize=(15, 8)) # 그래프 크기 설정
    bars = plt.bar(plot_names, plot_counts, color='skyblue', edgecolor='black')
    
    # 막대 위에 숫자 표시
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, height, '%d' % int(height), 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.xlabel('Class Name', fontsize=12)
    plt.ylabel('Number of Instances', fontsize=12)
    plt.title(f'Dataset Distribution (Total Images: {len(img_files)})', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45) # X축 글씨 45도 회전
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_dataset()

yolo yaml제작 / YOLO 포맷으로 변환하는 코드 

In [ ]:
import os
import json
import shutil
from tqdm import tqdm  # 진행률 표시 (없으면 pip install tqdm)

# =========================================================
# [설정] 경로 (사용자 환경)
# =========================================================

# 1. Training (학습용) 원본 경로
TRAIN_JSON_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\라벨링데이터"
TRAIN_IMAGE_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\1.Training\원천데이터"

# 2. Validation (검증용) 원본 경로
VAL_JSON_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\라벨링데이터"
VAL_IMAGE_DIR = r"E:\파이썬 공학경진대회\Dataset\141.지역사회 현안해결 데이터 장애인 길안내 자율주행 휠체어 융합센서 데이터\01.데이터\2.Validation\원천데이터"

# 3. 최종 저장될 YOLO 포맷 경로
OUTPUT_DIR = r"E:\파이썬 공학경진대회\YOLO_Safety_Dataset"

# =========================================================
# [설정] 타겟 클래스 (13개)
# 주의: 보내주신 JSON의 'bench'는 여기 없으므로 무시됩니다.
# =========================================================
CLASSES = [
    "bollard",          # 0
    "truck",            # 1
    "motorcycle",       # 2
    "baricade",         # 3
    "kickboard",        # 4
    "Warning Block",    # 5
    "car",              # 6
    "drainage",         # 7
    "manhole",          # 8
    "standing banner",  # 9
    "Leading Block",    # 10
    "labacon",          # 11
    "bicycle"           # 12
]

def convert_folder(json_dir, image_dir, split_name):
    """
    특정 폴더(Train 또는 Val)를 YOLO 포맷으로 변환하여 저장하는 함수
    """
    print(f"\n🚀 [{split_name}] 데이터셋 변환 시작...")
    
    # 저장 폴더 생성
    save_img_dir = os.path.join(OUTPUT_DIR, 'images', split_name)
    save_lbl_dir = os.path.join(OUTPUT_DIR, 'labels', split_name)
    os.makedirs(save_img_dir, exist_ok=True)
    os.makedirs(save_lbl_dir, exist_ok=True)

    # 1. 이미지 인덱싱
    print(f"   Step 1: {split_name} 이미지 목록 스캔 중...")
    image_map = {}
    for root, dirs, files in os.walk(image_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                name_no_ext = os.path.splitext(file)[0]
                image_map[name_no_ext] = os.path.join(root, file)
    
    print(f"      -> 이미지 {len(image_map)}장 발견")

    # 2. JSON 변환 및 저장
    print(f"   Step 2: JSON 변환 및 파일 복사 중...")
    processed_count = 0
    
    # 모든 JSON 파일 탐색
    json_files = []
    for root, dirs, files in os.walk(json_dir):
        for file in files:
            if file.lower().endswith('.json'):
                json_files.append(os.path.join(root, file))

    for json_path in tqdm(json_files, desc=f"Converting {split_name}"):
        try:
            filename = os.path.basename(json_path)
            name_no_ext = os.path.splitext(filename)[0]

            if name_no_ext not in image_map:
                continue

            img_src_path = image_map[name_no_ext]

            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # --- [수정된 핵심 파트] 문자열("1920")을 숫자(1920)로 강제 변환 ---
            img_w = data.get('imageWidth')
            img_h = data.get('imageHeight')
            
            # 값이 문자열이어도 int로 변환
            if img_w is not None:
                img_w = int(img_w)
            if img_h is not None:
                img_h = int(img_h)
            
            # 혹시라도 JSON에 크기가 없거나 0이면 이미지 파일 직접 열어서 확인
            if not img_w or not img_h:
                from PIL import Image
                with Image.open(img_src_path) as pil_img:
                    img_w, img_h = pil_img.size

            # --- 라벨링 정보 추출 ---
            yolo_lines = []
            shapes = data.get('shapes', [])
            
            for shape in shapes:
                label = shape.get('label')
                
                # 타겟 클래스 리스트에 있는 것만 처리
                if label in CLASSES:
                    cls_id = CLASSES.index(label)
                    points = shape.get('points')
                    
                    normalized_points = []
                    for x, y in points:
                        # 좌표값도 확실하게 실수(float)로 변환
                        x = float(x)
                        y = float(y)
                        
                        # Clipping (이미지 밖으로 나간 좌표 보정)
                        x = max(0, min(x, img_w))
                        y = max(0, min(y, img_h))
                        
                        normalized_points.append(f"{x / img_w:.6f}")
                        normalized_points.append(f"{y / img_h:.6f}")
                    
                    line = f"{cls_id} " + " ".join(normalized_points)
                    yolo_lines.append(line)

            # 유효한 타겟 라벨이 하나라도 있는 경우만 저장
            if yolo_lines:
                shutil.copy2(img_src_path, os.path.join(save_img_dir, os.path.basename(img_src_path)))
                
                txt_filename = name_no_ext + ".txt"
                with open(os.path.join(save_lbl_dir, txt_filename), 'w') as f:
                    f.write("\n".join(yolo_lines))
                
                processed_count += 1

        except Exception as e:
            # 에러가 나도 멈추지 않고 넘어가되 로그 출력
            print(f"⚠️ Error processing {json_path}: {e}")

    print(f"   ✨ [{split_name}] 완료: {processed_count}장 생성됨.")


def create_dataset_final():
    # 0. 출력 폴더 초기화
    if os.path.exists(OUTPUT_DIR):
        print("🗑️ 기존 데이터셋 폴더 정리 중...")
        try:
            shutil.rmtree(OUTPUT_DIR)
        except:
            print("⚠️ 기존 폴더 삭제 실패. 덮어쓰기 진행합니다.")

    # 1. Training 데이터 변환
    if os.path.exists(TRAIN_JSON_DIR):
        convert_folder(TRAIN_JSON_DIR, TRAIN_IMAGE_DIR, 'train')
    else:
        print("❌ Training 폴더 경로를 찾을 수 없습니다.")

    # 2. Validation 데이터 변환
    if os.path.exists(VAL_JSON_DIR):
        convert_folder(VAL_JSON_DIR, VAL_IMAGE_DIR, 'val')
    else:
        print("❌ Validation 폴더 경로를 찾을 수 없습니다.")

    # 3. data.yaml 생성
    yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

nc: {len(CLASSES)}
names: {CLASSES}
    """
    with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
        f.write(yaml_content)

    print("\n" + "="*50)
    print("🎉 데이터셋 구축 완료!")
    print(f"📂 저장 위치: {OUTPUT_DIR}")
    print("="*50)

if __name__ == "__main__":
    create_dataset_final()

원천 데이터(JSON)를 YOLO 학습용 포맷(.txt)으로 변환한 후, 데이터 손실이나 라벨링 오류가 발생하지 않았는지 최종 확인.
  - 학습에 사용될 `data.yaml` 설정 파일과 실제 라벨 파일들의 클래스 분포가 일치하는지 시각적으로 검증.

In [ ]:
import os
import glob
import yaml
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

# =========================================================
# [설정] 분석할 data.yaml 경로
# =========================================================
# 사용자님의 코드에 적힌 경로를 그대로 가져왔습니다.
YAML_PATH = r"E:\파이썬 공학경진대회\YOLO_Safety_Dataset_Fast\data.yaml"

def analyze_dataset():
    # 1. data.yaml 파일 읽기
    if not os.path.exists(YAML_PATH):
        print(f"❌ 파일 없음: {YAML_PATH}")
        return

    with open(YAML_PATH, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    # 클래스 이름 가져오기
    class_names = data.get('names', {})
    # 딕셔너리나 리스트 형태 둘 다 대응
    if isinstance(class_names, list):
        names_dict = {i: name for i, name in enumerate(class_names)}
    else:
        names_dict = class_names

    # 2. 라벨 폴더 경로 유추
    # data.yaml이 있는 폴더를 기준으로 labels/train 경로를 찾습니다.
    base_dir = os.path.dirname(YAML_PATH)
    
    # 보통 YOLO 구조: root/images/train  <->  root/labels/train
    label_dir = os.path.join(base_dir, "labels", "train")
    
    if not os.path.exists(label_dir):
        # 혹시 data.yaml 안의 경로가 절대경로일 경우 대비
        train_path_in_yaml = data.get('train', '')
        if os.path.isabs(train_path_in_yaml):
             # images/train -> labels/train 로 변경 시도
             label_dir = train_path_in_yaml.replace("images", "labels")
        else:
             label_dir = os.path.join(base_dir, train_path_in_yaml.replace("images", "labels"))

    if not os.path.exists(label_dir):
        print(f"❌ 라벨 폴더를 찾을 수 없습니다: {label_dir}")
        print("경로 구조가 표준 YOLO 형식이 아닌 것 같습니다.")
        return

    print(f"🚀 데이터 분석 시작...")
    print(f"📂 분석 대상: {label_dir}")

    # 3. 라벨 파일 스캔 및 카운트
    txt_files = glob.glob(os.path.join(label_dir, "*.txt"))
    print(f"📄 총 라벨 파일 수: {len(txt_files)}개")

    counter = Counter()

    for txt_file in tqdm(txt_files, desc="Counting Classes"):
        with open(txt_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    counter[class_id] += 1

    # 4. 결과 출력 (텍스트)
    print("\n" + "="*40)
    print(f"📊 클래스별 분포 (총 객체 수: {sum(counter.values())})")
    print("="*40)
    
    # 클래스 ID 순으로 정렬
    sorted_ids = sorted(names_dict.keys())
    
    labels_for_plot = []
    counts_for_plot = []

    for cls_id in sorted_ids:
        count = counter[cls_id]
        name = names_dict.get(cls_id, f"Unknown_{cls_id}")
        
        print(f"ID {cls_id:2d} [{name:<15}]: {count:5d} 개")
        
        labels_for_plot.append(name)
        counts_for_plot.append(count)

    print("="*40)

    # 5. 시각화 (그래프)
    try:
        plt.figure(figsize=(14, 7))
        bars = plt.bar(labels_for_plot, counts_for_plot, color='steelblue', edgecolor='black')
        
        # 막대 위에 숫자 표시
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2.0, height, 
                     f'{int(height)}', ha='center', va='bottom', fontweight='bold')

        plt.title(f"Dataset Distribution (Fast Ver.) - Total Objects: {sum(counter.values())}", fontsize=14)
        plt.xlabel("Class Name", fontsize=12)
        plt.ylabel("Count", fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
        print("📈 그래프가 표시되었습니다.")
        
    except Exception as e:
        print(f"⚠️ 그래프 출력 중 오류 발생 (텍스트 결과는 정상입니다): {e}")

if __name__ == "__main__":
    analyze_dataset()